# Phytclust Workflow

In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2
results_dir = "/home/ganesank/project/phytclust/results"  # change

## Importing 

In [ ]:
import pandas as pd


def sort_and_group_tsv(input_file, output_file):
    # Load the data
    df = pd.read_csv(input_file, sep="\t")

    # Convert chromosome column to integers for sorting
    df["chr"] = df["chr"].str.replace("chr", "").replace({"X": 23}).astype(int)

    # Sort by sample_id, chr, and start
    df = df.sort_values(by=["sample_id", "chr", "start"])

    # Convert chromosome column back to 'X' where applicable
    df["chr"] = df["chr"].replace({23: "X"}).astype(str)
    df["chr"] = "chr" + df["chr"]

    # Save the sorted DataFrame to a new TSV file
    df.to_csv(output_file, sep="\t", index=False)


# Example usage
input_file = "/home/ganesank/project/phytclust/Data/Roland_2015/processed_chunks/raw/OV03-08_sorted.tsv"
output_file = "/home/ganesank/project/phytclust/Data/Roland_2015/processed_chunks/grouped/OV03-08_sorted_grouped.tsv"
sort_and_group_tsv(input_file, output_file)

In [ ]:
tree_path = "/home/ganesank/project/phytclust/Data/william_medicc_trees/Multi_clonal_mouse/output_segments_all.tsv_final_tree.new"
tree = Phylo.read(tree_path, "newick", rooted=True)

In [ ]:
# with newick string
tree_string = (
"(((A:5, B:3)C1:6, (C:3, D:7)D1:4)A13:22, (((E:7, F:13)E12:5, G:6)B23:10, H:60):35):0;"
)
tree = Phylo.read(io.StringIO(tree_string), "newick")\

# With newick file
# tree_path = "/home/ganesank/project/phytclust/Data/ar53_r220.tree"
# tree = Phylo.read(tree_path, "newick", rooted=True)

In [ ]:
tree.count_terminals()

In [ ]:
clust_obj = PhytClust(tree, should_plot_scores=True, num_peaks=10, resolution_on=True)

In [ ]:
clust_obj.backtrack

In [ ]:
clust_obj.plot(n = 11, height_scale = 0.7, label_func=lambda x: "", marker_size=1000)

In [ ]:
from Bio import Phylo
from collections import defaultdict
clade_dict = clust_obj.clusters[6]
grouped_clades = defaultdict(list)
for clade_name, value in clade_dict.items():
    grouped_clades[value].append(clade_name)

# Find the MRCA for each group
mrca_names = {}
for value, clade_names in grouped_clades.items():
    # Find clades in the tree by name
    clades = [next(tree.find_clades(name=name), None) for name in clade_names]
    clades = [c for c in clades if c is not None]  # Remove None if a clade is not found

    # Find MRCA for the group
    if clades:
        mrca = tree.common_ancestor(clades)
        mrca_names[value] = mrca.name

print(mrca_names)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

array = clust_obj.den[0:100]
plt.figure(figsize=(10, 5))
plt.plot(array, marker="o", linestyle="-", color="b", label="beta_1/beta_k")
plt.xlabel("Index")
plt.ylabel("Value")
plt.title("Plot of an Array")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
clust_obj.plot(n = 28, height_scale = 0.2, label_func = lambda x: x.name, marker_size = 250)

In [ ]:
clust_obj.plot(n = 4, height_scale = 0.8, label_func = lambda x: "", marker_size = 1000)

In [ ]:
clust_obj.plot_scores(clust_obj.scores, k_start=0, k_end=100)

In [ ]:
from copy import deepcopy
from Bio import Phylo

# Assuming clust_obj.tree is your tree and mrcas contains clade names
tree = deepcopy(clust_obj.tree)
mrcas_name = list(mrcas.values())

def set_parent(clade, parent=None):
    clade.parent = parent
    for child in clade.clades:
        set_parent(child, clade)

set_parent(tree.root)

mrca_clades = []
for clade_name in mrcas_name:
    for clade in tree.find_clades(order="postorder"):
        if clade is not None:
            mrca_clades.append(clade)
        else:
            print(f"Clade with name {clade_name} not found in the tree")

clades_to_keep = set()
for clade in mrca_clades:
    while clade:
        clades_to_keep.add(clade)
        clade = clade.parent

for clade in tree.find_clades(order="postorder"):
    if clade not in clades_to_keep:
        try:
            tree.prune(clade)
        except ValueError as e:
            print(f"Error pruning clade: {clade.name}")
            print(e)


In [ ]:
import pandas as pd

input_file = "/home/ganesank/project/phytclust/Data/william_medicc_trees/BRCA_deficient_line/hscn.csv"
import medicc

cn_df = medicc.io.read_and_parse_input_data(
    input_file, total_copy_numbers=False, allele_columns=["cn_a", "cn_b"]
)
fig = medicc.plot.plot_cn_heatmap(
    cn_df, final_tree=tree, total_copy_numbers=False, alleles=["cn_a", "cn_b"]
)
%matplotlib inline

In [ ]:
fig

In [ ]:
# plot with labels
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 8})

label_func = lambda x: x.name if hasattr(x, "name") else x

clust_obj.plot(
    save=True,
    results_dir="../results",  # results_dir
    show_terminal_labels=True,
    outlier=False,
    top_n=1,
    label_func=label_func,
    width_scale=8,
    height_scale=0.4,
    marker_size=40,
)

In [ ]:
# plot for all k values
for n in range(1, len(tree.get_terminals())):
    clust_obj.plot(
        n=n,
        height_scale=0.4,
        outlier=False,
        save=False,
        results_dir="../results",
    )

In [ ]:
import os
import logging
import matplotlib.collections as mpcollections
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import logging
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Tuple, Union


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights


logger = logging.getLogger(__name__)


def plot_multiple_clusters(
    input_df: pd.DataFrame,
    final_tree: Optional[Any] = None,
    y_posns: Optional[Dict[str, int]] = None,
    cmax: Optional[int] = None,
    tree_width_ratio: float = 1,
    cbar_width_ratio: float = 0.05,
    figsize: Tuple[int, int] = (20, 10),
    tree_marker_size: int = 0,
    show_internal_nodes: bool = False,
    title: str = "",
    tree_label_func: Optional[callable] = None,
    cmap: str = "tab20b",
    outgroup: str = "diploid",
    fixed_x_range: Tuple[int, int] = (10000, 50000),
) -> plt.Figure:

    cmax = cmax or np.max(input_df["cluster_ID"].values.astype(int))
    sample_labels = input_df["leaf_name"].unique().tolist()
    required_columns = {"leaf_name", "comparison_IDs", "cluster_ID"}
    if not required_columns.issubset(input_df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_columns}")
        print(input_df.columns)

    # df_reset = input_df.reset_index()
    data_matrix = input_df.pivot(
        index="comparison_IDs", columns="leaf_name", values="cluster_ID"
    )

    data_matrix = data_matrix.reindex(columns=sample_labels).fillna(0)

    Z = data_matrix.T.values
    color_norm = mcolors.Normalize(vmin=0, vmax=cmax)

    if final_tree is None:
        fig, ax = plt.subplots(
            figsize=figsize,
            ncols=1,
            sharey=False,
            gridspec_kw={"width_ratios": [1]},
        )
        if not show_internal_nodes:
            logger.warning(
                'No tree provided, so "show_internal_nodes=False" is ignored'
            )
        y_posns = y_posns or {s: i for i, s in enumerate(sample_labels)}
        ax.set_title(
            title,
            x=0,
            y=1,
            ha="left",
            va="bottom",
            pad=20,
            fontweight="bold",
            fontsize=16,
            zorder=10,
        )
    else:
        fig, axs = plt.subplots(
            figsize=(figsize[0] * 0.5, figsize[1]),
            ncols=2,
            sharey=False,
            gridspec_kw={"width_ratios": [tree_width_ratio * 0.2, 0.5], "wspace": 0.05},
        )
        tree_ax, ax = axs

        y_posns = {
            k.name: v
            for k, v in _get_y_positions(
                final_tree, adjust=show_internal_nodes, outgroup=outgroup
            ).items()
        }
        plot_tree(
            final_tree,
            ax=tree_ax,
            outgroup=outgroup,
            label_func=tree_label_func or (lambda x: ""),
            hide_internal_nodes=True,
            show_branch_lengths=False,
            show_events=False,
            line_width=0.5,
            marker_size=tree_marker_size,
            title=title,
            label_colors=None,
        )
        tree_ax.set_axis_off()
        # fig.set_constrained_layout_pads(w_pad=0, h_pad=0, hspace=0.0, wspace=150)

    n_samples = len(sample_labels)
    n_comparisons = data_matrix.shape[0]
    x_pos = np.linspace(fixed_x_range[0], fixed_x_range[1], n_comparisons + 1)
    y_pos = np.arange(n_samples + 1) + 0.5

    # color_norm = mcolors.Normalize(vmin=0, vmax=cmax)
    im = ax.pcolormesh(x_pos, y_pos, Z, cmap=cmap, norm=color_norm)
    ax.set_yticks([])
    ax.set_ylim(n_samples + 0.5, 0.5)

    return fig


# fig = plot_multiple_clusters(
#     input_df=df_clusters, final_tree=tree, outgroup=None, tree_marker_size=10
# )

In [ ]:
clust_obj.best_peaks

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
clust_obj_clusters = [
    # clust_obj.clusters[clust_obj.best_peaks[2] - 1],
    clust_obj.clusters[11],
    # clust_obj.clusters[clust_obj.best_peaks[1] - 1],
]
comparison_ids = ["6", "8", "20", "21"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df_clusters = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])
# df_clusters.set_index(["leaf_name", "comparison_IDs"], inplace=True)
# Drop the extra `index` column after resetting the index
# df_clusters = df_clusters.reset_index(drop=True)

# Call the plot_multiple_clusters function with the custom colormaps
fig = plot_multiple_clusters(
    input_df=df_clusters, final_tree=clust_obj.tree, outgroup=None, tree_marker_size=10
)
#
# Show the plot
plt.show()

In [ ]:
clust_obj.best_peaks

In [ ]:
print(df_clusters.head())
print(df_clusters.index)  # Check if it's still an index
print(df_clusters.columns)  # Ensure required columns are present

In [ ]:
clust_obj.plot(n = 8, height_scale=0.8)

In [ ]:
from collections import Counter
from Bio.Phylo.BaseTree import Clade

print(Counter(clust_obj.clusters[7].values()))